In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from mini_nn.layers import Linear, ReLu
from mini_nn.optim import SGD
from mini_nn.losses import MSELoss

# End-to-End Training

## Problem Definition

We will train MiniNN to approximate:

\[
y = 3x_1 - 2x_2^{x_1} + \ln(x_3) + 0.5 + \epsilon
\]

where \(\epsilon \sim U(-1,1)\).


# Generate Data

In [ ]:
# --- set generator ---
gen = np.random.default_rng(42)

# --- create features ---
n = 10_000

x1 = np.arange(0, 1, 0.0001)
x2 = gen.uniform(0, 1, n)
x3 = gen.uniform(0.001, 5, n)

noise = gen.uniform(-1, 1, n)
constant = 0.5

# --- merge x and create y ---
x = np.column_stack([x1, x2, x3])

y = (
    3 * x1
    - 2 * (x2 ** x1)
    + np.log(x3)
    + constant
    + noise
)

y = y.reshape(-1, 1)

print(f'x shape: {x.shape}')
print(f'y shape: {y.shape}')

## Train / Validation Split

In [ ]:
# Random, reproducible 80:20 split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

print(f'x_train shape: {x_train.shape}')
print(f'y_train shape: {y_train.shape}')
print()
print(f'x_test shape: {x_test.shape}')
print(f'y_test shape: {y_test.shape}')

In [ ]:
plt.figure(figsize=(14, 4))

plt.scatter(
    range(len(y_train)),
    y_train,
    s=5,
    alpha=0.5,
    label='y train'
)

plt.scatter(
    range(len(y_train), len(y_train) + len(y_test)),
    y_test,
    s=5,
    alpha=0.5,
    label='y validation'
)

plt.title('Target values after random train / validation split')
plt.legend()
plt.show()

# Neural-Network Setup

In [ ]:
# --- input layer + activation ---
in_layer = Linear(
    in_features=x.shape[1],
    out_features=16,
    initialize='he_normal'
)
in_relu = ReLu()

# --- hidden layer + activation ---
linear1 = Linear(
    in_features=16,
    out_features=8,
    initialize='he_normal'
)
relu1 = ReLu()

# --- output layer ---
out_layer = Linear(
    in_features=8,
    out_features=1,
    initialize='he_uniform'
)

# --- optimizer ---
lr = 0.001
optimizer = SGD(
    layers=[in_layer, in_relu, linear1, relu1, out_layer],
    learning_rate=lr
)

# Use separate loss objects so validation never overwrites
# the training-loss cache.
train_loss_func = MSELoss()
val_loss_func = MSELoss()

## Forward / Backward Shape Check

This cell checks the information flow without updating the parameters.


In [ ]:
# --- forward ---
h1 = in_layer(x_train)
print('after in_layer:', h1.shape)

h1 = in_relu(h1)
print('after in_relu:', h1.shape)

h2 = linear1(h1)
print('after linear1:', h2.shape)

h2 = relu1(h2)
print('after relu1:', h2.shape)

y_pred = out_layer(h2)
print('after out_layer:', y_pred.shape)

loss = train_loss_func(y_train, y_pred)
print(f'loss: {loss:.6f}')

print()

# --- backward ---
grad = train_loss_func.backward()
print('after loss backward:', grad.shape)

grad = out_layer.backward(grad)
print('after out_layer backward:', grad.shape)

grad = relu1.backward(grad)
print('after relu1 backward:', grad.shape)

grad = linear1.backward(grad)
print('after linear1 backward:', grad.shape)

grad = in_relu.backward(grad)
print('after in_relu backward:', grad.shape)

grad = in_layer.backward(grad)
print('after in_layer backward:', grad.shape)

# Training Loop

In [ ]:
def train_model(
    x_train,
    y_train,
    x_test,
    y_test,
    train_loss_function,
    val_loss_function,
    optimizer,
    epochs,
    print_every=100,
    return_hist=True
):
    train_loss_hist = []
    val_loss_hist = []

    for epoch in range(epochs):

        # --- TRAIN FORWARD ---
        x_batch = in_layer(x_train)
        x_batch = in_relu(x_batch)
        x_batch = linear1(x_batch)
        x_batch = relu1(x_batch)
        y_pred = out_layer(x_batch)

        # --- TRAIN LOSS ---
        train_loss = train_loss_function(y_train, y_pred)
        train_loss_hist.append(train_loss)

        # --- BACKWARD ---
        grad = train_loss_function.backward()
        grad = out_layer.backward(grad)
        grad = relu1.backward(grad)
        grad = linear1.backward(grad)
        grad = in_relu.backward(grad)
        grad = in_layer.backward(grad)

        # --- VALIDATION ---
        # Validation forward overwrites the temporary forward caches,
        # but the training gradients (dw, db) have already been computed.
        x_val = in_layer(x_test)
        x_val = in_relu(x_val)
        x_val = linear1(x_val)
        x_val = relu1(x_val)
        val_pred = out_layer(x_val)

        val_loss = val_loss_function(y_test, val_pred)
        val_loss_hist.append(val_loss)

        # --- PARAMETER UPDATE ---
        # Uses the training gradients computed above.
        optimizer.step()

        if epoch % print_every == 0 or epoch == epochs - 1:
            print(
                f'epoch {epoch:4d} | '
                f'train loss={train_loss:.6f} | '
                f'val loss={val_loss:.6f}'
            )

    if return_hist:
        return train_loss_hist, val_loss_hist

In [ ]:
epochs = 2000

train_loss, val_loss = train_model(
    x_train=x_train,
    y_train=y_train,
    x_test=x_test,
    y_test=y_test,
    train_loss_function=train_loss_func,
    val_loss_function=val_loss_func,
    optimizer=optimizer,
    epochs=epochs,
    print_every=100
)

## Loss Curves

In [ ]:
plt.figure(figsize=(14, 4))

plt.plot(train_loss, label='train loss')
plt.plot(val_loss, label='validation loss')

plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Training and validation loss')
plt.legend()
plt.show()

## Final Evaluation

In [ ]:
# Recompute validation predictions after the final optimizer update.
x_val = in_layer(x_test)
x_val = in_relu(x_val)
x_val = linear1(x_val)
x_val = relu1(x_val)
final_val_pred = out_layer(x_val)

final_val_loss = val_loss_func(y_test, final_val_pred)

baseline_pred = np.full_like(y_test, y_train.mean())
baseline_mse = val_loss_func(y_test, baseline_pred)

print(f'baseline MSE: {baseline_mse:.6f}')
print(f'network MSE:  {final_val_loss:.6f}')
print()
print(f'prediction mean: {final_val_pred.mean():.6f}')
print(f'prediction std:  {final_val_pred.std():.6f}')
print(f'y_train mean:    {y_train.mean():.6f}')

In [ ]:
print('Sample predictions:')
for i in range(10):
    print(
        f'true={y_test[i, 0]: .4f} | '
        f'pred={final_val_pred[i, 0]: .4f}'
    )

# Side Experiment: Linear Sanity Check

This experiment uses separate variable names so it cannot overwrite the
main experiment's train / validation data or objects.

The true relationship is:

\[
y = 3x_1 - 2x_2 + 0.5
\]

A single linear layer should recover weights close to \(3\) and \(-2\),
with bias close to \(0.5\).


In [ ]:
sanity_gen = np.random.default_rng(42)
sanity_n = 1000

sanity_x1 = sanity_gen.uniform(-1, 1, sanity_n)
sanity_x2 = sanity_gen.uniform(-1, 1, sanity_n)

sanity_X = np.column_stack([sanity_x1, sanity_x2])

sanity_y = 3 * sanity_x1 - 2 * sanity_x2 + 0.5
sanity_y = sanity_y.reshape(-1, 1)

sanity_X_train, sanity_X_test, sanity_y_train, sanity_y_test = train_test_split(
    sanity_X,
    sanity_y,
    test_size=0.20,
    random_state=42
)

sanity_model = Linear(
    in_features=2,
    out_features=1,
    initialize='he_normal'
)

sanity_loss_func = MSELoss()

sanity_optimizer = SGD(
    layers=[sanity_model],
    learning_rate=0.05
)

In [ ]:
sanity_train_losses = []
sanity_val_losses = []

sanity_epochs = 1000

for epoch in range(sanity_epochs):

    # --- train forward ---
    sanity_pred = sanity_model(sanity_X_train)

    # --- train loss ---
    sanity_loss = sanity_loss_func(sanity_y_train, sanity_pred)
    sanity_train_losses.append(sanity_loss)

    # --- backward ---
    sanity_grad = sanity_loss_func.backward()
    sanity_model.backward(sanity_grad)

    # --- update ---
    sanity_optimizer.step()

    # --- validation ---
    sanity_val_pred = sanity_model(sanity_X_test)
    sanity_val_loss = np.mean((sanity_y_test - sanity_val_pred) ** 2)
    sanity_val_losses.append(sanity_val_loss)

print('Learned weights:')
print(sanity_model.weight)

print('\nLearned bias:')
print(sanity_model.bias)

print('\nFinal train loss:', sanity_train_losses[-1])
print('Final validation loss:', sanity_val_losses[-1])

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(sanity_train_losses, label='Train')
plt.plot(sanity_val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Linear sanity-check loss')
plt.legend()
plt.show()